# Large-Scale Experiment: Graph Generation

This notebook generates hundreds of random graphs with diverse structures and parameters for the large-scale embedding comparison experiment.

## Overview
- Generate 100 graphs per type (8 types = 800 total graphs)
- Vary graph sizes (100-500 nodes)
- Vary type-specific parameters
- Select seeds and targets for evaluation
- Save graphs with metadata
- Create manifest file for HPC processing

In [ ]:
import sys
import os
import yaml
import json
import numpy as np
import networkx as nx
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm
import itertools

# Add src to path
sys.path.insert(0, os.path.abspath('../src'))

from quvine.data.random_graphs import (
    generate_erdos_renyi,
    generate_barabasi_albert,
    generate_watts_strogatz,
    generate_powerlaw_cluster,
    generate_modular_network,
    generate_geometric_random,
    generate_regular_graph,
    generate_small_world
)

## 1. Load Configuration

In [ ]:
# Load experiment configuration
config_path = '../configs/large_scale_experiment.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print(f"Experiment: {config['experiment']['name']}")
print(f"Description: {config['experiment']['description']}")
print(f"\nGraph types: {config['graphs']['types']}")
print(f"Graphs per type: {config['graphs']['num_graphs_per_type']}")
print(f"Total graphs: {len(config['graphs']['types']) * config['graphs']['num_graphs_per_type']}")

## 2. Setup Output Directory

In [ ]:
# Create output directory structure
output_dir = Path(config['experiment']['output_dir'])
graphs_dir = output_dir / 'graphs'
metadata_dir = output_dir / 'metadata'

graphs_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {output_dir}")
print(f"Graphs directory: {graphs_dir}")
print(f"Metadata directory: {metadata_dir}")

## 3. Define Graph Generation Functions

In [ ]:
def generate_parameter_combinations(graph_type, config):
    """
    Generate all parameter combinations for a graph type.
    
    Returns list of (n_nodes, params_dict) tuples.
    """
    # Get size range
    min_nodes = config['graphs']['size']['min_nodes']
    max_nodes = config['graphs']['size']['max_nodes']
    node_step = config['graphs']['size']['node_step']
    node_sizes = list(range(min_nodes, max_nodes + 1, node_step))
    
    # Get type-specific parameters
    type_config = config['graphs'][graph_type]
    
    combinations = []
    
    if graph_type == 'erdos_renyi':
        for n in node_sizes:
            for p in type_config['p_range']:
                combinations.append((n, {'p': p}))
    
    elif graph_type == 'barabasi_albert':
        for n in node_sizes:
            for m in type_config['m_range']:
                if m < n:  # m must be less than n
                    combinations.append((n, {'m': m}))
    
    elif graph_type == 'watts_strogatz':
        for n in node_sizes:
            for k in type_config['k_range']:
                for p in type_config['p_range']:
                    if k < n:  # k must be less than n
                        combinations.append((n, {'k': k, 'p': p}))
    
    elif graph_type == 'powerlaw_cluster':
        for n in node_sizes:
            for m in type_config['m_range']:
                for p in type_config['p_range']:
                    if m < n:
                        combinations.append((n, {'m': m, 'p': p}))
    
    elif graph_type == 'modular_network':
        for n in node_sizes:
            for num_comm in type_config['num_communities_range']:
                for p_in in type_config['p_in_range']:
                    for p_out in type_config['p_out_range']:
                        if num_comm < n:  # Must have fewer communities than nodes
                            combinations.append((n, {
                                'num_communities': num_comm,
                                'p_in': p_in,
                                'p_out': p_out
                            }))
    
    elif graph_type == 'geometric_random':
        for n in node_sizes:
            for radius in type_config['radius_range']:
                combinations.append((n, {
                    'radius': radius,
                    'dim': type_config['dim']
                }))
    
    elif graph_type == 'regular_graph':
        for n in node_sizes:
            for d in type_config['d_range']:
                if d < n and (n * d) % 2 == 0:  # Regular graph constraints
                    combinations.append((n, {'d': d}))
    
    elif graph_type == 'small_world':
        for n in node_sizes:
            for k in type_config['k_range']:
                for p in type_config['p_range']:
                    if k < n:
                        combinations.append((n, {'k': k, 'p': p}))
    
    return combinations


def sample_combinations(combinations, num_samples, seed=42):
    """
    Sample a subset of parameter combinations.
    """
    rng = np.random.RandomState(seed)
    if len(combinations) <= num_samples:
        return combinations
    indices = rng.choice(len(combinations), size=num_samples, replace=False)
    return [combinations[i] for i in indices]


def generate_graph_with_params(graph_type, n_nodes, params, seed):
    """
    Generate a graph of the specified type with given parameters.
    """
    generators = {
        'erdos_renyi': generate_erdos_renyi,
        'barabasi_albert': generate_barabasi_albert,
        'watts_strogatz': generate_watts_strogatz,
        'powerlaw_cluster': generate_powerlaw_cluster,
        'modular_network': generate_modular_network,
        'geometric_random': generate_geometric_random,
        'regular_graph': generate_regular_graph,
        'small_world': generate_small_world
    }
    
    generator = generators[graph_type]
    return generator(n_nodes=n_nodes, seed=seed, **params)


def select_seeds_and_targets(G, config, seed):
    """
    Select seed and target nodes for evaluation.
    """
    rng = np.random.RandomState(seed)
    nodes = list(G.nodes())
    n_nodes = len(nodes)
    
    # Calculate number of seeds and targets
    eval_config = config['graphs']['evaluation']
    num_seeds = int(n_nodes * eval_config['num_seeds_ratio'])
    num_targets = int(n_nodes * eval_config['num_targets_ratio'])
    
    # Apply min/max constraints
    num_seeds = max(eval_config['min_seeds'], min(eval_config['max_seeds'], num_seeds))
    num_targets = max(eval_config['min_targets'], min(eval_config['max_targets'], num_targets))
    
    # Ensure we don't exceed available nodes
    num_seeds = min(num_seeds, n_nodes // 2)
    num_targets = min(num_targets, n_nodes - num_seeds)
    
    # Select seeds and targets (non-overlapping)
    selected = rng.choice(nodes, size=num_seeds + num_targets, replace=False)
    seeds = selected[:num_seeds].tolist()
    targets = selected[num_seeds:].tolist()
    
    return seeds, targets

## 4. Generate Graphs

In [ ]:
# Initialize manifest
manifest = []
generation_stats = {}

# Set random seed
base_seed = config['experiment']['seed']
np.random.seed(base_seed)

# Generate graphs for each type
for graph_type in tqdm(config['graphs']['types'], desc='Graph types'):
    print(f"\n{'='*60}")
    print(f"Generating {graph_type} graphs")
    print(f"{'='*60}")
    
    # Generate all parameter combinations
    all_combinations = generate_parameter_combinations(graph_type, config)
    print(f"Total possible combinations: {len(all_combinations)}")
    
    # Sample combinations to get desired number of graphs
    num_graphs = config['graphs']['num_graphs_per_type']
    combinations = sample_combinations(all_combinations, num_graphs, seed=base_seed)
    print(f"Selected combinations: {len(combinations)}")
    
    # Generate graphs
    type_stats = {
        'num_graphs': 0,
        'num_nodes': [],
        'num_edges': [],
        'num_seeds': [],
        'num_targets': [],
        'failed': 0
    }
    
    for idx, (n_nodes, params) in enumerate(tqdm(combinations, desc=f'{graph_type}')):
        try:
            # Generate unique seed for this graph
            graph_seed = base_seed + idx + len(manifest)
            
            # Generate graph
            G = generate_graph_with_params(graph_type, n_nodes, params, graph_seed)
            
            # Check if graph is connected (important for walks)
            if not nx.is_connected(G):
                # Take largest connected component
                largest_cc = max(nx.connected_components(G), key=len)
                G = G.subgraph(largest_cc).copy()
                # Relabel nodes to be sequential
                G = nx.convert_node_labels_to_integers(G)
            
            # Select seeds and targets
            seeds, targets = select_seeds_and_targets(G, config, graph_seed)
            
            # Create graph ID
            graph_id = f"{graph_type}_{idx:04d}"
            
            # Save graph
            graph_path = graphs_dir / f"{graph_id}.graphml"
            nx.write_graphml(G, graph_path)
            
            # Create metadata
            metadata = {
                'graph_id': graph_id,
                'graph_type': graph_type,
                'graph_path': str(graph_path.relative_to(output_dir)),
                'n_nodes': G.number_of_nodes(),
                'n_edges': G.number_of_edges(),
                'parameters': params,
                'seed': graph_seed,
                'seeds': seeds,
                'targets': targets,
                'num_seeds': len(seeds),
                'num_targets': len(targets),
                'generated_at': datetime.now().isoformat()
            }
            
            # Save metadata
            metadata_path = metadata_dir / f"{graph_id}_metadata.json"
            with open(metadata_path, 'w') as f:
                json.dump(metadata, f, indent=2)
            
            # Add to manifest
            manifest.append(metadata)
            
            # Update stats
            type_stats['num_graphs'] += 1
            type_stats['num_nodes'].append(G.number_of_nodes())
            type_stats['num_edges'].append(G.number_of_edges())
            type_stats['num_seeds'].append(len(seeds))
            type_stats['num_targets'].append(len(targets))
            
        except Exception as e:
            print(f"\nFailed to generate graph {idx}: {e}")
            type_stats['failed'] += 1
            continue
    
    # Store stats
    generation_stats[graph_type] = type_stats
    
    # Print summary
    print(f"\n{graph_type} Summary:")
    print(f"  Generated: {type_stats['num_graphs']} graphs")
    print(f"  Failed: {type_stats['failed']}")
    if type_stats['num_graphs'] > 0:
        print(f"  Nodes: {np.mean(type_stats['num_nodes']):.1f} ± {np.std(type_stats['num_nodes']):.1f}")
        print(f"  Edges: {np.mean(type_stats['num_edges']):.1f} ± {np.std(type_stats['num_edges']):.1f}")
        print(f"  Seeds: {np.mean(type_stats['num_seeds']):.1f} ± {np.std(type_stats['num_seeds']):.1f}")
        print(f"  Targets: {np.mean(type_stats['num_targets']):.1f} ± {np.std(type_stats['num_targets']):.1f}")

## 5. Save Manifest and Statistics

In [ ]:
# Save manifest
manifest_path = output_dir / 'graph_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"Saved manifest with {len(manifest)} graphs to {manifest_path}")

# Save generation statistics
stats_path = output_dir / 'generation_stats.json'
with open(stats_path, 'w') as f:
    json.dump(generation_stats, f, indent=2)

print(f"Saved generation statistics to {stats_path}")

# Save experiment configuration
config_copy_path = output_dir / 'experiment_config.yaml'
with open(config_copy_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Saved experiment configuration to {config_copy_path}")

## 6. Overall Statistics

In [ ]:
import pandas as pd

# Create summary dataframe
summary_data = []
for graph_type, stats in generation_stats.items():
    if stats['num_graphs'] > 0:
        summary_data.append({
            'Graph Type': graph_type,
            'Num Graphs': stats['num_graphs'],
            'Failed': stats['failed'],
            'Avg Nodes': f"{np.mean(stats['num_nodes']):.1f} ± {np.std(stats['num_nodes']):.1f}",
            'Avg Edges': f"{np.mean(stats['num_edges']):.1f} ± {np.std(stats['num_edges']):.1f}",
            'Avg Seeds': f"{np.mean(stats['num_seeds']):.1f} ± {np.std(stats['num_seeds']):.1f}",
            'Avg Targets': f"{np.mean(stats['num_targets']):.1f} ± {np.std(stats['num_targets']):.1f}"
        })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*80)
print("OVERALL SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))
print(f"\nTotal graphs generated: {len(manifest)}")
print(f"Total failed: {sum(stats['failed'] for stats in generation_stats.values())}")

# Save summary
summary_path = output_dir / 'generation_summary.csv'
summary_df.to_csv(summary_path, index=False)
print(f"\nSaved summary to {summary_path}")

## 7. Create HPC Job Array Mapping

In [ ]:
# Create a simple mapping file for HPC job arrays
# Each line: job_id,graph_id,graph_path,metadata_path

job_mapping = []
for job_id, entry in enumerate(manifest):
    graph_id = entry['graph_id']
    graph_path = entry['graph_path']
    metadata_path = f"metadata/{graph_id}_metadata.json"
    job_mapping.append(f"{job_id},{graph_id},{graph_path},{metadata_path}")

# Save job mapping
job_mapping_path = output_dir / 'job_mapping.csv'
with open(job_mapping_path, 'w') as f:
    f.write("job_id,graph_id,graph_path,metadata_path\n")
    f.write("\n".join(job_mapping))

print(f"Created job mapping for {len(job_mapping)} jobs")
print(f"Saved to {job_mapping_path}")
print(f"\nUse this file with HPC job arrays:")
print(f"  Job array range: 0-{len(job_mapping)-1}")
print(f"  Each job processes one graph")

## 8. Verification

In [ ]:
# Verify a few random graphs
print("Verifying random graphs...\n")

sample_indices = np.random.choice(len(manifest), size=min(5, len(manifest)), replace=False)

for idx in sample_indices:
    entry = manifest[idx]
    graph_path = output_dir / entry['graph_path']
    
    # Load graph
    G = nx.read_graphml(graph_path)
    
    print(f"Graph: {entry['graph_id']}")
    print(f"  Type: {entry['graph_type']}")
    print(f"  Nodes: {G.number_of_nodes()} (expected: {entry['n_nodes']})")
    print(f"  Edges: {G.number_of_edges()} (expected: {entry['n_edges']})")
    print(f"  Seeds: {entry['num_seeds']}")
    print(f"  Targets: {entry['num_targets']}")
    print(f"  Connected: {nx.is_connected(G)}")
    print()

## Summary

This notebook has generated all graphs for the large-scale experiment. The outputs include:

1. **Graphs**: Saved in GraphML format in `graphs/` directory
2. **Metadata**: Individual JSON files in `metadata/` directory
3. **Manifest**: Complete list of all graphs in `graph_manifest.json`
4. **Job Mapping**: CSV file for HPC job arrays in `job_mapping.csv`
5. **Statistics**: Generation statistics in `generation_stats.json`
6. **Configuration**: Copy of experiment config in `experiment_config.yaml`

### Next Steps

1. Run the pipeline script on each graph (can be parallelized on HPC)
2. Aggregate results from all jobs
3. Analyze results and create visualizations

See `scripts/run_single_graph_pipeline.py` for the processing pipeline.